Prueba

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [2]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\Desktop\\pruebas_collab\\datosNarmax\\24pasos_gru_consumption.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [3]:
datos.head()

,temp,zone1,zone2,zone3,hour,e
date,,,,,,
2017-01-01 00:00:00,-2.051356,0.151445,-0.851669,0.033500,-1.660858,NaN
2017-01-01 00:10:00,-2.074813,-0.433079,-0.211097,0.016502,-1.660858,NaN
2017-01-01 00:20:00,-2.091152,-0.527709,-0.283791,-0.055068,-1.660858,NaN
2017-01-01 00:30:00,-2.122213,-0.651648,-0.411186,-0.174054,-1.660858,NaN
2017-01-01 00:40:00,-2.154568,-0.774750,-0.507632,-0.244729,-1.660858,NaN


In [4]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Espacio de búsqueda

In [5]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'epochs': hp.choice('epochs', [2 ** i for i in range(3, 9)]),  # Número de épocas de entrenamiento
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [6]:
futuros = 24
pasados  = 12

In [7]:
datosX = []
datosY = []
for i in range(pasados, len(datos) - futuros + 1):
  datosX.append(datos.iloc[i-pasados:i, 0:datos.shape[1]])
  datosY.append(datos.iloc[i+futuros-1:i+futuros, 1])


In [8]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (52381, 12, 6)
Dimensiones de Y: (52381, 1)


In [9]:
print(datosX[0])

[[-2.05135573  0.15144479 -0.85166941  0.03349964 -1.66085756         nan]
 [-2.07481311 -0.43307937 -0.21109699  0.01650173 -1.66085756         nan]
 [-2.0911524  -0.52770864 -0.28379117 -0.05506842 -1.66085756         nan]
 [-2.12221321 -0.65164785 -0.41118591 -0.17405378 -1.66085756         nan]
 [-2.15456823 -0.77474965 -0.50763164 -0.2447293  -1.66085756         nan]
 [-2.16556893 -0.87272862 -0.59759968 -0.29303915 -1.66085756         nan]
 [-2.19986525 -0.95898362 -0.68109001 -0.32166721 -1.51634012         nan]
 [-2.22332263 -1.03518949 -0.74658674 -0.39681586 -1.51634012         nan]
 [-2.19387957 -1.12730648 -0.83223632 -0.46391287 -1.51634012         nan]
 [-2.22413151 -1.19597551 -0.88909611 -0.49969795 -1.51634012         nan]
 [-2.22008713 -1.24873342 -0.98842083 -0.52385287 -1.51634012         nan]
 [-2.22736701 -1.29730419 -1.03232523 -0.5614272  -1.51634012         nan]]


Se dividen nuevamente los conjuntos de datos

In [10]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (36666, 12, 6)
Las dimensiones de testX son:  (10529, 12, 6)
Las dimensiones de valX son:  (5186, 12, 6)


In [11]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (36666, 1)
Las dimensiones de testY son:  (10529, 1)
Las dimensiones de valY son:  (5186, 1)


Se crean métricas para medir desempeño

In [12]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [13]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [14]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(testX.shape[1], testX.shape[2])))
    if (params['layers'] == 1):
      model.add(GRU(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(GRU(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(GRU(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(testX, testY, epochs=params['epochs'],
                        validation_split=0.3,
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [15]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=50, trials=trials, rstate=np.random.default_rng(42))

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

231/231 - 82s - 355ms/step - ia: 0.2296 - loss: 0.8105 - mae: 0.7335 - rmse: 0.8935 - smape: 1.5531 - val_ia: 0.2392 - val_loss: 0.9107 - val_mae: 0.7916 - val_rmse: 0.8493 - val_smape: 1.5747

Epoch 2/128                                           

231/231 - 5s - 23ms/step - ia: 0.4833 - loss: 0.5268 - mae: 0.5787 - rmse: 0.7186 - smape: 1.1259 - val_ia: 0.3583 - val_loss: 0.3579 - val_mae: 0.4904 - val_rmse: 0.5469 - val_smape: 0.9060

Epoch 3/128                                           

231/231 - 5s - 23ms/step - ia: 0.6318 - loss: 0.3569 - mae: 0.4719 - rmse: 0.5918 - smape: 0.8686 - val_ia: 0.3922 - val_loss: 0.2600 - val_mae: 0.4157 - val_rmse: 0.4683 - val_smape: 0.7971

Epoch 4/128                                           

231/231 - 5s - 21ms/step - ia: 0.6702 - loss: 0.3006 - mae: 0.4307 - rmse: 0.5439 - smape: 0.8014 - val_ia: 0.4139 - val_loss: 0.2346 - val_mae: 0.3921 - val_rmse: 0.4416 - val_smape: 0.8067

Epoch 

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/16                                                                         

29/29 - 58s - 2s/step - ia: 0.2652 - loss: 0.6394 - mae: 0.6493 - rmse: 0.7928 - smape: 1.4612 - val_ia: 0.5538 - val_loss: 0.3405 - val_mae: 0.4914 - val_rmse: 0.5670 - val_smape: 0.8941

Epoch 2/16                                                                         

29/29 - 6s - 195ms/step - ia: 0.7522 - loss: 0.2081 - mae: 0.3522 - rmse: 0.4521 - smape: 0.6431 - val_ia: 0.7311 - val_loss: 0.1726 - val_mae: 0.3274 - val_rmse: 0.4070 - val_smape: 0.6834

Epoch 3/16                                                                         

29/29 - 5s - 180ms/step - ia: 0.8254 - loss: 0.1115 - mae: 0.2580 - rmse: 0.3324 - smape: 0.5228 - val_ia: 0.7414 - val_loss: 0.1534 - val_mae: 0.3081 - val_rmse: 0.3823 - val_smape: 0.6572

Epoch 4/16                                                                         

29/29 - 5s - 168ms/step - ia: 0.8538 - loss: 0.0777 - mae: 0.2142 - rmse: 0.2783 - smape:

...

58/58 - 3s - 45ms/step - ia: 0.2212 - loss: 1.0003 - mae: 0.7985 - rmse: 0.9991 - smape: 1.5170 - val_ia: 0.3086 - val_loss: 1.0310 - val_mae: 0.8249 - val_rmse: 1.0031 - val_smape: 1.5940

Epoch 2/16                                                                            

58/58 - 0s - 8ms/step - ia: 0.2171 - loss: 1.0000 - mae: 0.8009 - rmse: 0.9987 - smape: 1.5348 - val_ia: 0.3108 - val_loss: 1.0232 - val_mae: 0.8221 - val_rmse: 0.9992 - val_smape: 1.5926

Epoch 3/16                                                                            

58/58 - 0s - 8ms/step - ia: 0.2176 - loss: 0.9847 - mae: 0.7935 - rmse: 0.9912 - smape: 1.5279 - val_ia: 0.3128 - val_loss: 1.0150 - val_mae: 0.8192 - val_rmse: 0.9951 - val_smape: 1.5893

Epoch 4/16                                                                            

58/58 - 0s - 8ms/step - ia: 0.2130 - loss: 0.9825 - mae: 0.7943 - rmse: 0.9901 - smape: 1.5305 - val_ia: 0.3149 - val_loss: 1.0077 - val_mae: 0.8166 - val_rmse: 0.9915 

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



29/29 - 5s - 184ms/step - ia: 0.2847 - loss: 1.2969 - mae: 0.9109 - rmse: 1.1071 - smape: 1.4428 - val_ia: 0.3580 - val_loss: 1.1944 - val_mae: 0.9239 - val_rmse: 1.0605 - val_smape: 1.7164

Epoch 2/128                                                                           

29/29 - 1s - 45ms/step - ia: 0.2874 - loss: 0.7984 - mae: 0.7334 - rmse: 0.8919 - smape: 1.4369 - val_ia: 0.4320 - val_loss: 0.7036 - val_mae: 0.6921 - val_rmse: 0.8116 - val_smape: 1.2569

Epoch 3/128                                                                           

29/29 - 1s - 47ms/step - ia: 0.5521 - loss: 0.4643 - mae: 0.5445 - rmse: 0.6756 - smape: 1.0091 - val_ia: 0.6893 - val_loss: 0.2667 - val_mae: 0.4159 - val_rmse: 0.5067 - val_smape: 0.7255

Epoch 4/128                                                                           

29/29 - 1s - 46ms/step - ia: 0.7107 - loss: 0.2724 - mae: 0.4136 - rmse: 0.5202 - smape: 0.7323 - val_ia: 0.7533 - val_loss: 0.1627 - val_mae: 0.3214 - val_rmse: 0.3

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



29/29 - 2s - 85ms/step - ia: 0.3015 - loss: 3.3399 - mae: 1.5698 - rmse: 1.7735 - smape: 1.4525 - val_ia: 0.3375 - val_loss: 2.3131 - val_mae: 1.3290 - val_rmse: 1.4857 - val_smape: 1.6604

Epoch 2/128                                                                          

29/29 - 0s - 6ms/step - ia: 0.2785 - loss: 0.9822 - mae: 0.8212 - rmse: 0.9873 - smape: 1.4421 - val_ia: 0.3636 - val_loss: 1.0661 - val_mae: 0.8719 - val_rmse: 1.0018 - val_smape: 1.7185

Epoch 3/128                                                                          

29/29 - 0s - 6ms/step - ia: 0.2512 - loss: 0.7692 - mae: 0.7261 - rmse: 0.8761 - smape: 1.4903 - val_ia: 0.3747 - val_loss: 0.8799 - val_mae: 0.7852 - val_rmse: 0.9098 - val_smape: 1.5801

Epoch 4/128                                                                          

29/29 - 0s - 6ms/step - ia: 0.2876 - loss: 0.7129 - mae: 0.6991 - rmse: 0.8439 - smape: 1.4369 - val_ia: 0.3964 - val_loss: 0.8189 - val_mae: 0.7525 - val_rmse: 0.8767 - v

In [16]:
print(best)

{'activation': 1, 'batch': 5, 'dropout': 0.30000000000000004, 'epochs': 4, 'layers': 2.0, 'learning_rate': 0.006238028275242282, 'units': 4}
